/Volumes/workspace/default/capstoneproject/S&P500_Data_Raw.csv

In [0]:
%sql
create or replace temp view spx_raw
using csv
options (
  path = "/Volumes/workspace/default/capstoneproject/S&P500_Data_Raw.csv",
  header = true,
  inferSchema = "true"
  );

In [0]:
%sql
select * from spx_raw;

In [0]:
from pyspark.sql import functions as F

path = "/Volumes/workspace/default/capstoneproject/S&P500_Data_Raw.csv"

df_raw = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(path))

display(df_raw)
print(df_raw.columns)


In [0]:
rename_map = {
  "('Date',_'')": "Date",
  "('Open',_'^GSPC')": "Open",
  "('High',_'^GSPC')": "High",
  "('Low',_'^GSPC')": "Low",
  "('Close',_'^GSPC')": "Close",
  "('Adj_Close',_'^GSPC')": "Adj_Close",
  "('Volume',_'^GSPC')": "Volume",
}

df = df_raw
for old, new in rename_map.items():
    if old in df.columns:
        df = df.withColumnRenamed(old, new)

df = (df
      .withColumn("Date", F.to_date("Date"))
      .withColumn("Open", F.col("Open").cast("double"))
      .withColumn("High", F.col("High").cast("double"))
      .withColumn("Low", F.col("Low").cast("double"))
      .withColumn("Close", F.col("Close").cast("double"))
      .withColumn("Adj_Close", F.col("Adj_Close").cast("double"))
      .withColumn("Volume", F.col("Volume").cast("double"))
      .orderBy("Date")
)

display(df)
print(df.columns)


In [0]:
%sql
DESCRIBE spx_raw_clean;

In [0]:
from pyspark.sql.types import *
from pyspark.sql import functions as F

schema = StructType([
    StructField("Date", DateType()),
    StructField("Adj_Close", DoubleType()),
    StructField("Close", DoubleType()),
    StructField("High", DoubleType()),
    StructField("Low", DoubleType()),
    StructField("Open", DoubleType()),
    StructField("Volume", DoubleType()),
])

df_typed = (spark.read
            .schema(schema)
            .option("header", True)
            .csv("/Volumes/workspace/default/capstoneproject/S&P500_Data_Raw.csv"))

df_typed.createOrReplaceTempView("capstone_typed")

display(df_typed)

In [0]:
display(
  df_typed.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("Date").isNull(), 1).otherwise(0)).alias("null_dates"),
    F.sum(F.when(F.col("Adj_Close") .isNull(), 1).otherwise(0)).alias("null_adj_close"),
    F.sum(F.when(F.col("Close").isNull(), 1).otherwise(0)).alias("null_close"),
    F.sum(F.when(F.col("High").isNull(), 1).otherwise(0)).alias("null_high"),
    F.sum(F.when(F.col("Low").isNull(), 1).otherwise(0)).alias("null_low"),
    F.sum(F.when(F.col("Open").isNull(), 1).otherwise(0)).alias("null_open"),
    F.sum(F.when(F.col("Volume").isNull(), 1).otherwise(0)).alias("null_volume")
  )
)

In [0]:
%sql
SELECT
  Date,
  COUNT(*) AS cnt
FROM capstone_typed
GROUP BY Date
HAVING COUNT(*) > 1;


In [0]:
%sql
SELECT
  Date, Adj_Close, Close, High, Low, Open, Volume,
  COUNT(*) AS cnt
FROM capstone_typed
GROUP BY
  Date, Adj_Close, Close, High, Low, Open, Volume
HAVING COUNT(*) > 1
ORDER BY cnt DESC;


In [0]:
(df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("spx_raw_clean"))


In [0]:
%sql
--Added features; SMA, volatility, OBV, VWAP, CMF
CREATE OR REPLACE TABLE spx_window_indicators AS
WITH x AS (
  SELECT
    *,
    LAG(Adj_Close) OVER (ORDER BY Date) AS prev_close,
    (Adj_Close / LAG(Adj_Close) OVER (ORDER BY Date) - 1.0) AS ret_1d,
    LN(Adj_Close / LAG(Adj_Close) OVER (ORDER BY Date)) AS log_ret_1d
  FROM spx_raw_clean
)
SELECT
  *,
  -- Simple moving averages
  AVG(Adj_Close) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)  AS SMA_20,
  AVG(Adj_Close) OVER (ORDER BY Date ROWS BETWEEN 49 PRECEDING AND CURRENT ROW)  AS SMA_50,
  AVG(Adj_Close) OVER (ORDER BY Date ROWS BETWEEN 199 PRECEDING AND CURRENT ROW) AS SMA_200,

  -- Rolling volatility (20d) and annualized
  STDDEV_SAMP(ret_1d) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS vol_20d,
  STDDEV_SAMP(ret_1d) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) * SQRT(252) AS vol_20d_ann,

  -- Volume SMA
  AVG(Volume) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS Vol_SMA_20,

  -- OBV
  SUM(
    CASE
      WHEN Adj_Close > prev_close THEN Volume
      WHEN Adj_Close < prev_close THEN -Volume
      ELSE 0
    END
  ) OVER (ORDER BY Date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS OBV,

  -- VWAP (cumulative)
  SUM(((High + Low + Close)/3.0) * Volume) OVER (ORDER BY Date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
  / NULLIF(SUM(Volume) OVER (ORDER BY Date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 0) AS VWAP,

  -- CMF(20)
  (
    SUM(
      (
        ((Close - Low) - (High - Close)) / NULLIF((High - Low), 0)
      ) * Volume
    ) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW)
  )
  / NULLIF(
    SUM(Volume) OVER (ORDER BY Date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW),
    0
  ) AS CMF_20

FROM x
ORDER BY Date;


In [0]:
%sql
SELECT Date, Adj_Close, SMA_20, ret_1d, vol_20d, OBV, VWAP, CMF_20
FROM spx_window_indicators
ORDER BY Date DESC
LIMIT 10;


In [0]:
import pandas as pd
import numpy as np
from pyspark.sql.types import *
from pyspark.sql import functions as F

#Using python to calculate the EMA, MACD and RSI

base = spark.table("spx_raw_clean").select("Date", "Adj_Close").orderBy("Date")

schema = StructType([
    StructField("Date", DateType()),
    StructField("EMA_12", DoubleType()),
    StructField("EMA_26", DoubleType()),
    StructField("MACD", DoubleType()),
    StructField("MACD_signal", DoubleType()),
    StructField("MACD_hist", DoubleType()),
    StructField("RSI_14", DoubleType()),
])

def add_recursive(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("Date").copy()
    price = pdf["Adj_Close"]

    ema12 = price.ewm(span=12, adjust=False).mean()
    ema26 = price.ewm(span=26, adjust=False).mean()

    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    macd_hist = macd - macd_signal

    delta = price.diff()
    gain = delta.clip(lower=0)
    loss = (-delta).clip(lower=0)

    avg_gain = gain.ewm(alpha=1/14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/14, adjust=False).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - (100 / (1 + rs))

    return pd.DataFrame({
        "Date": pdf["Date"],
        "EMA_12": ema12.astype(float),
        "EMA_26": ema26.astype(float),
        "MACD": macd.astype(float),
        "MACD_signal": macd_signal.astype(float),
        "MACD_hist": macd_hist.astype(float),
        "RSI_14": rsi.astype(float),
    })

def iterator_fn(iterator):
    pdf = pd.concat(list(iterator), ignore_index=True)
    yield add_recursive(pdf)

recursive = base.mapInPandas(iterator_fn, schema=schema)
display(recursive.orderBy(F.desc("Date")))


In [0]:
windowed = spark.table("spx_window_indicators")
final = windowed.join(recursive, on="Date", how="left")

(final.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("spx_with_indicators"))


In [0]:
data_path = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/spx_with_indicators"

(final.write
 .format("delta")
 .mode("overwrite")
 .save(data_path))


In [0]:
csv_dir = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/exports/spx_with_indicators_csv"

(final
 .coalesce(1)
 .write
 .mode("overwrite")
 .option("header", True)
 .csv(csv_dir))


In [0]:
files = dbutils.fs.ls(csv_dir)
csv_part = [f.path for f in files if f.name.endswith(".csv")][0]

final_csv_path = "dbfs:/Volumes/workspace/default/capstoneproject/S&P500_Data_Clean/exports/S&P500_with_indicators.csv"

dbutils.fs.mv(csv_part, final_csv_path)
dbutils.fs.rm(csv_dir, recurse=True)

print("CSV saved to:", final_csv_path)
